In [ ]:
import ollama
import json
from Error import Error

prompt_config_path = "prompts.json"
host = "http://localhost:11434" # Ollama binds to port 11434 by default
model = "gemma3:1b"
available_error_types = "INFINITE_LOOP_ERROR, OFF_BY_ONE_ERROR, ARITHMETIC_ERROR"

# input from the changes in mined repo's pull request
code = "" \
        "counter = 0:\n" \
        "while counter < 5:\n" \
        "\tprint(counter)" 

In [2]:
# get the prompts from the json file

try:
    with open(prompt_config_path, "r", encoding="utf-8") as file:
        prompts = json.load(file)

        system_prompt_error = prompts['system_prompt_error']
        print(f"System prompt error fetched: {system_prompt_error}")

        system_prompt_suggestion = prompts['system_prompt_suggestion']
        print(f"System prompt suggestion fetched: {system_prompt_suggestion}")

        get_error_prompt = prompts['get_error_prompt']
        print(f"Get error prompt fetched: {get_error_prompt}")

        suggestion_prompt = prompts['suggestion_prompt']
        print(f"Suggestion prompt fetched: {suggestion_prompt}")

        print(f"\n'{prompt_config_path}' successfully loaded!")
        
except FileNotFoundError:
    print(f"Error: The file '{prompt_config_path}' was not found.")
except json.JSONDecodeError as e:
    print(f"Error decoding JSON: {e}")

client = ollama.Client(host=host)

System prompt error fetched: You are an expert software engineer doing python code review. When given a code sample and a list of possible errors, find if any of the errors exist in the code. Always return your response as a Python list of strings in this format: Error Type | Error severity (low/medium/high/severe) | Error description. If no errors are found return: NONE
System prompt suggestion fetched: You are an expert software engineer doing python code review. Suggest a fix for an error when you are given an error type, error severity level, and error description.
Get error prompt fetched: Here is the code sample {code} and the list of possible errors {available_error_types}
Suggestion prompt fetched: Here is the error type: {error_type}, severity level: {severity} and description: {error_description}. This is the code snippet: {code}.

'prompts.json' successfully loaded!


In [3]:
# get error prompt

def request_error(get_error_prompt: str) -> str:
    get_error_prompt = get_error_prompt.format(
        code = code, # the code in string format containing the changes in a specific pull request mined from the repo
        available_error_types = available_error_types
    )

    get_error_response = client.chat(
        model = model,
        messages = [
            {"role": "system", "content": system_prompt_error},
            {"role": "user", "content": get_error_prompt}
        ]
    )

    error_response = get_error_response["message"]["content"]
    return error_response

def parse_error_response(error_response: str) -> list[Error]:
    # will parse the response from llm
    if error_response == "NONE":
        return []
    
    parsed_errors = error_response.split("\n")[2:] # end result or parsing the error response from LLM: a list of error information in consistent format

    print(parsed_errors)

    # Create a list of Error objects. Assign error type, severity level, and error description generated from llm
    errors = []

    for idx, error_info in enumerate(parsed_errors):
        error_info_split = error_info.split("|")

        if (len(error_info_split[0]) != 0):

            error_type = error_info_split[0].strip()
            error_severity_level = error_info_split[1].strip()
            error_description = error_info_split[2].strip()

            error = Error(error_type, error_severity_level, error_description)
            errors.append(error)

    return errors

error_response = request_error(get_error_prompt)

errors = parse_error_response(error_response)

['INFINITE_LOOP_ERROR | severe | The `while` loop condition will never be met, resulting in an infinite loop.', 'OFF_BY_ONE_ERROR | low | The value of `counter` will never reach 5.', 'ARITHMETIC_ERROR | medium | The `print(counter)` statement is outside the loop and will always print the value of `counter`.', '']


In [ ]:
# get fix suggestion

def request_suggestion(error: Error) -> Error:
    error_type = error.get_error_type()
    severity = error.get_error_severity_level()
    error_description = error.get_error_description()

    global suggestion_prompt

    suggestion_prompt = suggestion_prompt.format(
        error_type = error_type,
        severity = severity,
        error_description = error_description,
        code = code
    )

    suggestion_response = client.chat(
        model = model,
        messages = [
            {"role": "system", "content": system_prompt_suggestion},
            {"role": "user", "content": suggestion_prompt}
        ]
    )

    suggestion_response = suggestion_response["message"]["content"]

    error.set_fix_suggestion(suggestion_response)

    return error_response

def get_all_fix_suggestions(errors: list[Error]) -> list[Error]:
    for idx, error in enumerate(errors):
        errors[idx] = request_suggestion(error)

    return errors

errors = get_all_fix_suggestions(errors)

errors

Okay, let's analyze this and provide a fix.

**Review & Suggested Solution**

The error is a classic example of a `while` loop condition never being met, leading to an infinite loop.  The provided code snippet *does* satisfy the condition `counter < 5` (the loop terminates when `counter` is 4), therefore, the loop doesn't run.

**Proposed Solution:**

The most straightforward fix is to simply remove the `print(counter)` line.  The loop condition is already correctly set so that the loop will *always* terminate.

**Corrected Code:**

```python
counter = 0
while counter < 5:
    print(counter)
```

**Explanation:**

By removing the `print(counter)` line, the loop effectively stops after the first iteration, thus avoiding the infinite loop.

**Further Considerations (Depending on context):**

* **Why is this happening?**  It's crucial to understand *why* this loop is generating an infinite loop.  The code *should* be designed to iterate a certain number of times. If this is a larger, more

['Error Type | Error severity (low/medium/high/severe) | Error description\n------- | -------- | --------\nINFINITE_LOOP_ERROR | severe | The `while` loop condition will never be met, resulting in an infinite loop.\nOFF_BY_ONE_ERROR | low | The value of `counter` will never reach 5.\nARITHMETIC_ERROR | medium | The `print(counter)` statement is outside the loop and will always print the value of `counter`.\n',
 'Error Type | Error severity (low/medium/high/severe) | Error description\n------- | -------- | --------\nINFINITE_LOOP_ERROR | severe | The `while` loop condition will never be met, resulting in an infinite loop.\nOFF_BY_ONE_ERROR | low | The value of `counter` will never reach 5.\nARITHMETIC_ERROR | medium | The `print(counter)` statement is outside the loop and will always print the value of `counter`.\n',
 'Error Type | Error severity (low/medium/high/severe) | Error description\n------- | -------- | --------\nINFINITE_LOOP_ERROR | severe | The `while` loop condition will ne